In [7]:
# ============================================================
# 参数配置
# ============================================================
param = {
    "test": "power",              # ['type1error', 'power']
    "sample_size": 200,           # [200, 400, 600, 800, 1000]
    "batch_size": 128,            # [32, 64, 128, 256]
    "z_dim": 1,                   # [5, 50, 250]
    "dx": 1,
    "dy": 1,
    "n_test": 100,                # [200, 2000] number of trails to run
    "epochs_num": 100,            # [1000, 1500] number of epoch
    "eps_std": 1.0,               # epsilon std
    "dist_z": 'gaussian',         # ['laplace', 'gaussian']
    "alpha_x": 0.20,              # only used under alternative [0.05, 0.10, 0.15, 0.20, 0.25]
    "m_value": 100,               # [100, 200]
    "k_value": 8,                 # [2, 4, 6, 8] fold number
    "j_value": 1000,              # [1000, 2000]
    "noise_dimension": 5,         # [5, 10, 20]
    "noise_dimension_type": "normal",   # ["normal", "unif", "Cauchy"]
    "noise_dimension_var": 1,     # [1, 4, 9, 16]
    "hidden_layer_size": 1024,    # [64, 128, 256, 512, 1024]
    "normal_ini": False,
    "preprocess": 'scale',        # ['normalize', 'scale', 'None']
    "G_lr": 7e-5,
    "alpha": 0.1,
    "alpha1": 0.05,
    "set_seeds": 0,
    "using_orcale": False,
    "lambda_1": 1,
    "lambda_2": 1,
    "using_Gen": '1',
    "boor_rv_type": 'gaussian',
    "wgt_decay": 1e-5,
    "lambda_3": 1e-4,
    "lambda_4": 2e-5,
    "drop_out_p": 0.2,
    "is_sparse": True,
    "sparse_ratio": 0.05,
    "lambda_mean": 0.3,
    "mean_samples": 20
}

import torch
import torch.distributions as TD
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from datetime import datetime
import functools
from tqdm import tqdm

# ============================================================
# [修改] 删除全局 device 变量，改为在 mGAN() 内部按 gpu_id 动态创建。
# 原代码：
#   enable_cuda = True
#   device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')
# ============================================================


def generate_samples_random(size=1000, sType='H0', dx=1, dy=1, dz=1,
                            nstd=1.0, alpha_x=0.05,
                            preprocess="None", dist_z='mean_nonlinear'):
    """
    Safer nonlinear conditional-mean DGP for testing mean alignment.

    H0:
        X = mu_X(Z) + sigma * eps_X
        Y = mu_Y(Z) + sigma * eps_Y

    H1:
        X = mu_X(Z) + sigma * eps_X
        Y = mu_Y(Z) + sigma * [Delta * eps_X + (1 - Delta) * eps_Y]

    This keeps the conditional marginals X|Z and Y|Z the same under H0 and H1,
    but introduces conditional dependence through shared residual shocks.
    """

    num = size

    if dist_z in ['mean_nonlinear', 'uniform']:
        Z = 2.0 * torch.rand(num, dz) - 1.0
    elif dist_z == 'gaussian':
        z_generator = TD.MultivariateNormal(torch.zeros(dz), torch.eye(dz))
        Z = z_generator.sample((num,))
    elif dist_z == 'laplace':
        laplace = TD.Laplace(torch.zeros(dz), torch.ones(dz))
        Z = laplace.sample((num,))
    else:
        raise ValueError(f"Unsupported dist_z: {dist_z}")

    z1 = Z[:, 0:1]

    # safer conditional means
    mu_x = 1.2 * torch.sin(2.0 * np.pi * z1) + 0.5 * z1
    mu_y = 1.2 * torch.cos(2.0 * np.pi * z1) - 0.5 * z1

    sigma_x = torch.ones(num, 1) * 0.8
    sigma_y = torch.ones(num, 1) * 0.8

    eps_x = torch.randn(num, 1)
    eps_y = torch.randn(num, 1)

    if sType == 'H0':
        eps_y_star = eps_y

    elif sType == 'H1':
        delta = torch.bernoulli(torch.ones(num, 1) * alpha_x)
        eps_y_star = delta * eps_x + (1.0 - delta) * eps_y

    else:
        raise ValueError("sType must be 'H0' or 'H1'.")

    X = mu_x + sigma_x * nstd * eps_x
    Y = mu_y + sigma_y * nstd * eps_y_star

    if dx > 1:
        X = X.repeat(1, dx)
    if dy > 1:
        Y = Y.repeat(1, dy)

    return X.float(), Y.float(), Z.float()
    
def generate_samples_from_fixed_Z_random(Z, size=1000, sType='H0',
                                          dx=1, dy=1, dz=1,
                                          nstd=1.0, alpha_x=0.05,
                                          normalize=True, seed=None,
                                          dist_z='mean_nonlinear'):
    """
    Oracle conditional generator for the safer nonlinear mean DGP.
    Always generates independent samples from P_{X|Z} and P_{Y|Z}.
    """

    if seed is not None:
        torch.manual_seed(seed)

    if Z.dim() == 1:
        Z = Z.reshape(-1, 1)

    num = Z.shape[0]
    device = Z.device

    z1 = Z[:, 0:1]

    mu_x = 1.2 * torch.sin(2.0 * np.pi * z1) + 0.5 * z1
    mu_y = 1.2 * torch.cos(2.0 * np.pi * z1) - 0.5 * z1

    sigma_x = torch.ones(num, 1, device=device) * 0.8
    sigma_y = torch.ones(num, 1, device=device) * 0.8

    eps_x = torch.randn(num, 1, device=device)
    eps_y = torch.randn(num, 1, device=device)

    X = mu_x + sigma_x * nstd * eps_x
    Y = mu_y + sigma_y * nstd * eps_y

    if dx > 1:
        X = X.repeat(1, dx)
    if dy > 1:
        Y = Y.repeat(1, dy)

    return X.float(), Y.float()
    
def get_p_value_stat_1(boot_num, M, n, gen_x_all_torch, gen_y_all_torch,
                        x_torch, y_torch, z_torch, sigma_w,
                        sigma_u=1, sigma_v=1, boor_rv_type="gaussian",
                        device=None):  # [修改] 新增 device 参数
    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1),
        ord=1, dim=2)
    w_mx = torch.exp(-w_mx / sigma_w)

    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(torch.exp(-torch.abs(
        gen_y_all_torch.repeat(n, 1, 1) - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u), dim=2)
    u_mx_3 = u_mx_2.T
    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(torch.exp(-torch.abs(
        gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)

    v_mx_1 = torch.exp(-torch.abs(x_torch.repeat(1, n) - x_torch.repeat(1, n).T) / sigma_v)
    v_mx_2 = torch.mean(torch.exp(-torch.abs(
        gen_x_all_torch.repeat(n, 1, 1) - x_torch.repeat(1, n).reshape(n, n, 1)) / sigma_v), dim=2)
    v_mx_3 = v_mx_2.T
    gen_x_all_torch_rep = gen_x_all_torch.repeat(n, 1, 1)
    temp2_mx = gen_x_all_torch_rep[:, :, 0].T
    sum2_mx = torch.mean(torch.exp(-torch.abs(
        gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)

    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        temp_add_mx = torch.mean(torch.exp(-torch.abs(
            gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)
        sum_mx = sum_mx + temp_add_mx
        temp2_mx = gen_x_all_torch_rep[:, :, i].T
        temp2_add_mx = torch.mean(torch.exp(-torch.abs(
            gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)
        sum2_mx = sum2_mx + temp2_add_mx

    u_mx_4 = 1 / M * sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4
    v_mx_4 = 1 / M * sum2_mx
    v_mx = v_mx_1 - v_mx_2 - v_mx_3 + v_mx_4

    FF_mx = u_mx * v_mx * w_mx * (1 - torch.eye(n).to(device))  # [修改]
    stat = 1 / (n - 1) * torch.sum(FF_mx).item()

    boottemp = np.array([])
    if boor_rv_type == "rademacher":
        eboot = torch.sign(torch.randn(n, boot_num)).to(device)  # [修改]
    elif boor_rv_type == "gaussian":
        eboot = torch.randn(n, boot_num).to(device)              # [修改]
    for bb in range(boot_num):
        random_mx = torch.matmul(eboot[:, bb].reshape(-1, 1), eboot[:, bb].reshape(-1, 1).T)
        bootmatrix = FF_mx * random_mx
        stat_boot = 1 / (n - 1) * torch.sum(bootmatrix).item()
        boottemp = np.append(boottemp, stat_boot)
    return stat, boottemp


class DatasetSelect(Dataset):
    def __init__(self, X, Y, Z):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index]


class DatasetSelect_GAN(torch.utils.data.Dataset):
    def __init__(self, X, Y, Z, batch_size):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return (self.X_real[index], self.Y_real[index], self.Z_real[index],
                self.Z_real[(self.batch_size + index) % self.sample_size])


class DatasetSelect_GAN_ver2(torch.utils.data.Dataset):
    def __init__(self, Y, Z, batch_size):
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = Z.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.Y_real[index], self.Z_real[index]


##### Auxilliary functions #####

def sample_noise(sample_size, noise_dimension, noise_type, input_var=1,
                 device=None):  # [修改] 新增 device 参数
    if noise_type == "normal":
        noise_generator = TD.MultivariateNormal(
            torch.zeros(noise_dimension).to(device),          # [修改]
            input_var * torch.eye(noise_dimension).to(device) # [修改]
        )
        Z = noise_generator.sample((sample_size,))
    if noise_type == "unif":
        Z = torch.rand(sample_size, noise_dimension)
    if noise_type == "Cauchy":
        Z = TD.Cauchy(torch.tensor([0.0]), torch.tensor([1.0])).sample(
            (sample_size, noise_dimension)).squeeze(2)
    return Z


##### GAN architecture #####

class Generator(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef, drop_out_p, drop_input=False):
        super(Generator, self).__init__()
        self.BN_type = BN_type
        self.ReLU_coef = ReLU_coef
        self.fc1 = torch.nn.Linear(input_dimension + noise_dimension, hidden_layer_size, bias=True)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN2 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN3 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
        self.leakyReLU1 = torch.nn.LeakyReLU(ReLU_coef)
        self.fc2 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc3 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc_last = torch.nn.Linear(hidden_layer_size, output_dimension, bias=True)
        self.sigmoid = torch.nn.Sigmoid()
        self.drop_out0 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out1 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out2 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out3 = torch.nn.Dropout(p=drop_out_p)
        self.drop_input = drop_input

    def forward(self, x):
        if self.BN_type:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.BN1(self.fc1(x))))
            x = self.drop_out2(self.leakyReLU1(self.BN2(self.fc2(x))))
            # x = self.drop_out3(self.leakyReLU1(self.BN3(self.fc3(x))))
            x = self.fc_last(x)
        else:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.fc1(x)))
            x = self.drop_out2(self.leakyReLU1(self.fc2(x)))
            # x = self.drop_out3(self.leakyReLU1(self.fc3(x)))
            x = self.fc_last(x)
        return x


class NonFullyConnected_1(torch.nn.Module):
    def __init__(self, size_in, size_out, m, bias=True):
        super(NonFullyConnected_1, self).__init__()
        # [修改] 不在此处 .to(device)，由外层调用者统一 .to(device)
        self.linear = torch.nn.Linear(m * size_in, m * size_out, bias=bias)
        self.mask = functools.reduce(
            torch.block_diag,
            [torch.ones(size_out, size_in) for i in range(m)])

    def forward(self, x):
        # [修改] mask 动态跟随模型所在设备
        self.linear.weight.data *= self.mask.to(self.linear.weight.device)
        return self.linear(x)


class Generator_2(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef,
                 hidden_layer_depth=1, ntargets_k=5):
        super(Generator_2, self).__init__()
        self.input_dimension = input_dimension + noise_dimension
        self.output_dimension = output_dimension
        self.ntargets_k = ntargets_k
        self.hidden_layer_sizes = [hidden_layer_size] * hidden_layer_depth
        self.BN_type = BN_type
        self.leakyrelu = torch.nn.LeakyReLU(ReLU_coef)
        self.linear_layers_from_input = torch.nn.Linear(
            self.input_dimension, ntargets_k * self.hidden_layer_sizes[0])
        self.linear_layers_between = torch.nn.ModuleList([
            NonFullyConnected_1(self.hidden_layer_sizes[0], self.hidden_layer_sizes[0], ntargets_k)
            for i in range(len(self.hidden_layer_sizes))
        ])
        self.linear8 = torch.nn.Linear(
            self.hidden_layer_sizes[0] * ntargets_k, self.output_dimension)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)

    def forward(self, input):
        if self.BN_type:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(self.BN1(output))
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(self.BN1(output))
        else:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(output)
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(output)
        return self.linear8(output)


##### Training procedures #####

def find_loss(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-torch.abs(Y.repeat(1, n) - Y.repeat(1, n).T) / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    mx_1_2 = torch.exp(-mx_1_2 / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2_1 = torch.exp(-torch.abs(Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y)
    mx_2 = mx_2_1 * mx_1_2
    mx_3 = mx_2.T
    mx_4_1 = torch.exp(-torch.abs(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y)
    mx_4 = mx_4_1 * mx_1_2
    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss = 1 / (n ** 2) * torch.sum(FF_mx)
    return loss


def find_loss_2(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-(Y.repeat(1, n) - Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=2, dim=2)
    mx_1_2 = torch.exp(-(mx_1_2 ** 2) / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2_1 = torch.exp(-(Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_2 = mx_2_1 * mx_1_2
    mx_3 = mx_2.T
    mx_4_1 = torch.exp(-(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_4 = mx_4_1 * mx_1_2
    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss = 1 / (n ** 2) * torch.sum(FF_mx)
    return loss


def train_ver3(
        G_zx, G_zy,
        X, Y, Z, X_test, Y_test, Z_test, M,
        noise_dimension, noise_type, G_lr, hidden_layer_size,
        DataLoader, BN_type, ReLU_coef,
        lambda_mean=0.5, mean_samples=20,
        epochs_num=50, patience=5, min_delta=1e-5,
        sigma_z=1, sigma_x=1, sigma_y=1, normal_ini=False,
        lambda_1=1, lambda_2=1, using_Gen='1', wgt_decay=0,
        lambda_3=0, drop_out_p=0.2, noise_dimension_var=1, lambda_4=0,
        device=None):  # [修改] 新增 device 参数
    input_dimension = Z.shape[1]
    output_dimension_y = Y.shape[1]
    output_dimension_x = X.shape[1]

    if G_zy is None or G_zx is None:
        if using_Gen == '1':
            G_zy = Generator(input_dimension, output_dimension_y, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)  # [修改]
            G_zx = Generator(input_dimension, output_dimension_x, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)  # [修改]
        elif using_Gen == '2':
            G_zy = Generator_2(input_dimension, output_dimension_y, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)             # [修改]
            G_zx = Generator_2(input_dimension, output_dimension_x, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)             # [修改]
        if normal_ini:
            for p in G_zy.parameters():
                p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))  # [修改]
            for p in G_zx.parameters():
                p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))  # [修改]

    # 无论是否是新模型，每一折都重新定义优化器以清空动量状态
    G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
    G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

    iter_count = 0
    G_zy = G_zy.train()
    G_zx = G_zx.train()

    best_loss = float('inf')
    counter = 0

    for epoch in range(epochs_num):
        batch_count = 0
        G_zy = G_zy.train()
        G_zx = G_zx.train()

        for X_real, Y_real, Z_real, Z_fake in DataLoader:
            X_real = X_real.to(device)   # [修改]
            Y_real = Y_real.to(device)   # [修改]
            Z_real = Z_real.to(device)   # [修改]
            Z_fake = Z_fake.to(device)   # [修改]

            batch_size = Z_real.shape[0]
            Z_repeated = Z_real.repeat_interleave(mean_samples, dim=0)

            Noise_for_mean = sample_noise(Z_repeated.shape[0], noise_dimension, noise_type,
                                          input_var=noise_dimension_var, device=device).to(device)  # [修改]

            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type,
                                      input_var=noise_dimension_var, device=device).to(device)       # [修改]
            Y_fake = G_zy(torch.cat((Z_real, Noise_fake), dim=1)).to(device)
            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type,
                                      input_var=noise_dimension_var, device=device).to(device)       # [修改]
            X_fake = G_zx(torch.cat((Z_real, Noise_fake), dim=1)).to(device)

            Y_generated_group = G_zy(torch.cat((Z_repeated, Noise_for_mean), dim=1))
            Y_generated_reshaped = Y_generated_group.reshape(batch_size, mean_samples, -1)
            Y_mean_pred = torch.mean(Y_generated_reshaped, dim=1)
            loss_mean_y = torch.nn.functional.mse_loss(Y_mean_pred, Y_real)

            X_generated_group = G_zx(torch.cat((Z_repeated, Noise_for_mean), dim=1))
            X_mean_pred = torch.mean(X_generated_group.reshape(batch_size, mean_samples, -1), dim=1)
            loss_mean_x = torch.nn.functional.mse_loss(X_mean_pred, X_real)

            g_zy_error = None
            G_zy_solver.zero_grad()
            g_zx_error = None
            G_zx_solver.zero_grad()

            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0
            for name, param in G_zy.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param, ord=1)

            mmd_loss_y = (lambda_1 * find_loss(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y) +
                          lambda_2 * find_loss_2(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y) +
                          lambda_3 * l1_regularization_rest_layers +
                          lambda_4 * l1_regularization_first_layer)
            g_zy_error = mmd_loss_y + lambda_mean * loss_mean_y
            g_zy_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zy.parameters(), max_norm=0.5)
            G_zy_solver.step()

            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0
            for name, param in G_zx.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param, ord=1)

            mmd_loss_x = (lambda_1 * find_loss(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x) +
                          lambda_2 * find_loss_2(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x) +
                          lambda_3 * l1_regularization_rest_layers +
                          lambda_4 * l1_regularization_first_layer)
            g_zx_error = mmd_loss_x + lambda_mean * loss_mean_x
            g_zx_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zx.parameters(), max_norm=0.5)
            G_zx_solver.step()

            iter_count += 1
            batch_count += 1

        current_loss = g_zx_error + g_zy_error
        if current_loss < best_loss - min_delta:
            best_loss = current_loss
            counter = 0
        else:
            counter += 1
        if counter >= patience:
            break

    return G_zy, G_zx


def mGAN(n=500, z_dim=1, simulation='type1error', batch_size=64, epochs_num=1000,
         nstd=1.0, z_dist='gaussian', x_dims=1, y_dims=1, a_x=0.05, M=500, k=2, boot_num=1000,
         noise_dimension=10, hidden_layer_size=512, normal_ini=False, preprocess='normalize',
         G_lr=1e-5, using_orcale=False, lambda_1=1, lambda_2=1,
         lambda_mean=0.3, mean_samples=20, using_Gen='1',
         boor_rv_type="gaussian", wgt_decay=0, lambda_3=1, drop_out_p=0.2,
         noise_dimension_var=1, noise_dimension_type="normal", lambda_4=1,
         gpu_id=0):
    """
    修正版 mGAN 函数。

    主要修正：
    1. 正确使用 median heuristic 计算 Laplacian kernel bandwidth：
       sigma_w = median_{i != j} ||Z_i - Z_j||_1
       sigma_u = median_{i != j} ||Y_i - Y_j||_1
       sigma_v = median_{i != j} ||X_i - X_j||_1

    2. 不再使用 median(exp(-distance)) 作为 bandwidth。

    3. 对 p-value 计算加入 tie correction：
       p = (1 + #{bootstrap_stat >= observed_stat}) / (B + 1)
       这样当 bootstrap statistic 与 observed statistic 出现大量 tie 时，
       不会错误地产生 p-value = 0。
    """

    # ============================================================
    # 0. 内部 helper：正确的 median L1 bandwidth
    # ============================================================

    def median_l1_bandwidth(A, min_bw=1e-6):
        """
        A: tensor with shape n x d
        return: median_{i != j} ||A_i - A_j||_1

        注意：
        原来的代码错误地使用 median(exp(-distance))。
        这里使用的是 pairwise distance 的 median。
        """

        n_local = A.shape[0]

        dist = torch.linalg.vector_norm(
            A.repeat(n_local, 1, 1) - torch.swapaxes(A.repeat(n_local, 1, 1), 0, 1),
            ord=1,
            dim=2
        )

        mask = ~torch.eye(n_local, dtype=torch.bool, device=A.device)
        vals = dist[mask]
        vals = vals[vals > 0]

        if vals.numel() == 0:
            return min_bw

        bw = torch.median(vals).item()

        if (not np.isfinite(bw)) or bw < min_bw:
            bw = min_bw

        return bw

    # ============================================================
    # 1. 设置 device
    # ============================================================

    enable_cuda = True

    if torch.cuda.is_available() and enable_cuda:
        device = torch.device(f'cuda:{gpu_id}')
    else:
        device = torch.device('cpu')

    # ============================================================
    # 2. 生成数据
    # ============================================================

    if simulation == 'type1error':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n,
            sType='H0',
            dx=x_dims,
            dy=y_dims,
            dz=z_dim,
            nstd=nstd,
            alpha_x=a_x,
            dist_z=z_dist,
            preprocess=preprocess
        )

    elif simulation == 'power':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n,
            sType='H1',
            dx=x_dims,
            dy=y_dims,
            dz=z_dim,
            nstd=nstd,
            alpha_x=a_x,
            dist_z=z_dist,
            preprocess=preprocess
        )

    else:
        raise ValueError('Test does not exist. simulation must be type1error or power.')

    x = sim_x.to(device)
    y = sim_y.to(device)
    z = sim_z.to(device)

    # ============================================================
    # 3. 修正后的 bandwidth 计算
    # ============================================================
    # 原来错误写法：
    #
    #   u_mx = torch.exp(-torch.abs(y.repeat(1, n) - y.repeat(1, n).T))
    #   sigma_u = torch.median(u_mx).item()
    #
    #   v_mx = torch.exp(-torch.abs(x.repeat(1, n) - x.repeat(1, n).T))
    #   sigma_v = torch.median(v_mx).item()
    #
    # 现在改成 pairwise distance 的 median。

    sigma_w = median_l1_bandwidth(z)
    sigma_u = median_l1_bandwidth(y)
    sigma_v = median_l1_bandwidth(x)

    # 如果你想 debug，可以临时打开这行：
    # print(f"sigma_w={sigma_w:.6f}, sigma_u={sigma_u:.6f}, sigma_v={sigma_v:.6f}")

    # ============================================================
    # 4. 初始化 cross-fitting 容器
    # ============================================================

    test_size = int(n / k)

    stat_all = torch.zeros(k, 1)
    boot_temp_all = torch.zeros(k, boot_num)

    cur_k = 0

    # ============================================================
    # 5. 初始化 generator
    # ============================================================
    # 保持你原来的写法：generator 在 fold 外初始化，fold 间继续训练。
    # 严格 cross-fitting 下建议每个 fold 重新初始化 generator；
    # 但本函数根据你的要求只重点修正 bandwidth 问题。

    if not using_orcale:
        G_zy = Generator(
            z_dim,
            y_dims,
            noise_dimension,
            hidden_layer_size,
            False,
            0.1,
            drop_out_p
        ).to(device)

        G_zx = Generator(
            z_dim,
            x_dims,
            noise_dimension,
            hidden_layer_size,
            False,
            0.1,
            drop_out_p
        ).to(device)
    else:
        G_zy = None
        G_zx = None

    # ============================================================
    # 6. Cross-fitting loop
    # ============================================================

    for k_fold in range(k):

        k_fold_start = int(n / k * k_fold)
        k_fold_end = int(n / k * (k_fold + 1))

        X_test = x[k_fold_start:k_fold_end]
        Y_test = y[k_fold_start:k_fold_end]
        Z_test = z[k_fold_start:k_fold_end]

        X_train = torch.cat((x[0:k_fold_start], x[k_fold_end:]))
        Y_train = torch.cat((y[0:k_fold_start], y[k_fold_end:]))
        Z_train = torch.cat((z[0:k_fold_start], z[k_fold_end:]))

        if k == 1:
            X_train, Y_train, Z_train = X_test, Y_test, Z_test

        train_xyz = DatasetSelect_GAN(X_train, Y_train, Z_train, batch_size)

        DataLoader_xyz = torch.utils.data.DataLoader(
            train_xyz,
            batch_size=batch_size,
            shuffle=True
        )

        # ========================================================
        # 6.1 训练 generator
        # ========================================================

        if not using_orcale:

            G_zy, G_zx = train_ver3(
                G_zx=G_zx,
                G_zy=G_zy,
                X=X_train,
                Y=Y_train,
                Z=Z_train,
                M=M,
                X_test=X_test,
                Y_test=Y_test,
                Z_test=Z_test,
                noise_dimension=noise_dimension,
                noise_type=noise_dimension_type,
                G_lr=G_lr,
                hidden_layer_size=hidden_layer_size,
                DataLoader=DataLoader_xyz,
                BN_type=False,
                ReLU_coef=0.1,
                lambda_mean=lambda_mean,
                mean_samples=mean_samples,
                epochs_num=epochs_num,
                sigma_z=sigma_w,
                sigma_x=sigma_v,
                sigma_y=sigma_u,
                normal_ini=normal_ini,
                lambda_1=lambda_1,
                lambda_2=lambda_2,
                using_Gen=using_Gen,
                wgt_decay=wgt_decay,
                lambda_3=lambda_3,
                drop_out_p=drop_out_p,
                noise_dimension_var=noise_dimension_var,
                lambda_4=lambda_4,
                device=device
            )

        # ========================================================
        # 6.2 在 test fold 上生成 fake samples
        # ========================================================

        dataset_test = DatasetSelect(X_test, Y_test, Z_test)

        dataloader_test = DataLoader(
            dataset_test,
            batch_size=1,
            shuffle=True
        )

        gen_x_all = torch.zeros(test_size, M, device=device)
        gen_y_all = torch.zeros(test_size, M, device=device)

        z_all = torch.zeros(test_size, z_dim, device=device)
        x_all = torch.zeros(test_size, x_dims, device=device)
        y_all = torch.zeros(test_size, y_dims, device=device)

        cur_itr = 0

        if not using_orcale:
            G_zx = G_zx.eval()
            G_zy = G_zy.eval()

        for i, (x_test, y_test, z_test) in enumerate(dataloader_test):

            x_test = x_test.to(device)
            y_test = y_test.to(device)
            z_test = z_test.to(device)

            z_test_temp = z_test.repeat(M, 1)

            if not using_orcale:

                z_test_temp = z_test_temp.to(device)

                Noise_fake_x = sample_noise(
                    z_test_temp.size()[0],
                    noise_dimension,
                    noise_dimension_type,
                    input_var=noise_dimension_var,
                    device=device
                ).to(device)

                with torch.no_grad():
                    fake_x = G_zx(
                        torch.cat((z_test_temp, Noise_fake_x), dim=1)
                    ).reshape(1, -1)

                Noise_fake_y = sample_noise(
                    z_test_temp.size()[0],
                    noise_dimension,
                    noise_dimension_type,
                    input_var=noise_dimension_var,
                    device=device
                ).to(device)

                with torch.no_grad():
                    fake_y = G_zy(
                        torch.cat((z_test_temp, Noise_fake_y), dim=1)
                    ).reshape(1, -1)

            elif using_orcale:

                if simulation == 'type1error':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(
                        z_test_temp,
                        size=M,
                        sType='H0',
                        dx=x_dims,
                        dy=y_dims,
                        dz=z_dim,
                        nstd=nstd,
                        alpha_x=a_x,
                        dist_z=z_dist
                    )

                elif simulation == 'power':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(
                        z_test_temp,
                        size=M,
                        sType='H1',
                        dx=x_dims,
                        dy=y_dims,
                        dz=z_dim,
                        nstd=nstd,
                        alpha_x=a_x,
                        dist_z=z_dist
                    )

                fake_x = fake_x.to(device).reshape(1, -1)
                fake_y = fake_y.to(device).reshape(1, -1)

            gen_x_all[cur_itr, :] = fake_x.detach().reshape(-1)
            gen_y_all[cur_itr, :] = fake_y.detach().reshape(-1)

            x_all[cur_itr, :] = x_test
            y_all[cur_itr, :] = y_test
            z_all[cur_itr, :] = z_test

            cur_itr = cur_itr + 1

        # ========================================================
        # 6.3 计算 statistic 和 bootstrap statistic
        # ========================================================

        cur_stat, cur_boot_temp = get_p_value_stat_1(
            boot_num,
            M,
            test_size,
            gen_x_all.to(device),
            gen_y_all.to(device),
            x_all.to(device),
            y_all.to(device),
            z_all.to(device),
            sigma_w,
            sigma_u,
            sigma_v,
            boor_rv_type,
            device=device
        )

        stat_all[cur_k, :] = cur_stat
        boot_temp_all[cur_k, :] = torch.from_numpy(cur_boot_temp)

        cur_k = cur_k + 1

    # ============================================================
    # 7. Oracle diagnostic
    # ============================================================

    if using_orcale:
        gen_x_mean = torch.mean(gen_x_all, dim=1).reshape(-1, 1)
        gen_y_mean = torch.mean(gen_y_all, dim=1).reshape(-1, 1)

        mse_x = torch.mean((gen_x_mean - x_all) ** 2).item()
        mse_y = torch.mean((gen_y_mean - y_all) ** 2).item()

        print(f'Test MSE x [{mse_x}], MSE y [{mse_y}]')

    # ============================================================
    # 8. p-value
    # ============================================================
    # 原代码：
    #
    #   return np.mean(torch.mean(boot_temp_all, dim=0).numpy() > torch.mean(stat_all).item())
    #
    # 这里顺手改成 tie-corrected p-value。
    # 如果 bootstrap statistic 和 observed statistic 大量相等，
    # 原来的严格 > 会把 p-value 错误算成 0。

    boot_stats = torch.mean(boot_temp_all, dim=0).detach().cpu().numpy()
    obs_stat = torch.mean(stat_all).item()

    p_value = (1.0 + np.sum(boot_stats >= obs_stat)) / (len(boot_stats) + 1.0)

    return p_value
    
# ============================================================
# 并行部分
# ============================================================
from joblib import Parallel, delayed
import multiprocessing
import numpy as np
from datetime import datetime


def run_experiment(params):
    # 1. 提取参数
    test                 = params["test"]
    sample_size          = params["sample_size"]
    batch_size           = params["batch_size"]
    z_dim                = params["z_dim"]
    dx                   = params["dx"]
    dy                   = params["dy"]
    n_test               = params["n_test"]
    epochs_num           = params["epochs_num"]
    eps_std              = params["eps_std"]
    dist_z               = params["dist_z"]
    alpha_x              = params["alpha_x"]
    m_value              = params["m_value"]
    k_value              = params["k_value"]
    j_value              = params["j_value"]
    noise_dimension      = params["noise_dimension"]
    noise_dimension_type = params["noise_dimension_type"]
    noise_dimension_var  = params["noise_dimension_var"]
    hidden_layer_size    = params["hidden_layer_size"]
    normal_ini           = params["normal_ini"]
    preprocess           = params["preprocess"]
    G_lr                 = params["G_lr"]
    alpha                = params["alpha"]
    alpha1               = params["alpha1"]
    set_seeds            = params["set_seeds"]
    using_orcale         = params["using_orcale"]
    lambda_1             = params["lambda_1"]
    lambda_2             = params["lambda_2"]
    using_Gen            = params["using_Gen"]
    boor_rv_type         = params["boor_rv_type"]
    wgt_decay            = params["wgt_decay"]
    lambda_3             = params["lambda_3"]
    lambda_4             = params["lambda_4"]
    drop_out_p           = params["drop_out_p"]
    lambda_mean          = params["lambda_mean"]
    mean_samples         = params["mean_samples"]

    np.random.seed(set_seeds)
    torch.manual_seed(set_seeds)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(set_seeds)

    # 2. [修改] 获取可用 GPU 数量，用于轮询分配
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    num_cores = min(28, n_test)
    if num_cores < 1:
        num_cores = 1

    print(f"[{datetime.now().strftime('%H:%M:%S')}] 开始并行实验...")
    print(f"模式: {test}, 样本量: {sample_size}, 交叉验证折数: {k_value}, "
          f"模型均值对齐超参数: {lambda_mean}, 实验次数: {n_test}, "
          f"并行核数: {num_cores}, 可用GPU数: {num_gpus}")
    if test == 'power':
        print(f"备择假设H_1下的模型参数 alpha_x: {alpha_x}")

    # 3. [修改] 按 job_index % num_gpus 轮询分配 GPU
    #    job 0,4,8,...  → GPU 0
    #    job 1,5,9,...  → GPU 1
    #    job 2,6,10,... → GPU 2
    #    job 3,7,11,... → GPU 3
    p_values = Parallel(n_jobs=num_cores)(
        delayed(mGAN)(
            n=sample_size, z_dim=z_dim, simulation=test, batch_size=batch_size,
            epochs_num=epochs_num, nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy,
            a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
            noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size,
            normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr,
            using_orcale=using_orcale, lambda_1=lambda_1, lambda_2=lambda_2,
            lambda_mean=lambda_mean, mean_samples=mean_samples,
            using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay,
            lambda_3=lambda_3, drop_out_p=drop_out_p,
            noise_dimension_var=noise_dimension_var,
            noise_dimension_type=noise_dimension_type, lambda_4=lambda_4,
            gpu_id=job_index % num_gpus    # [修改] 轮询分配 GPU
        )
        for job_index, _ in enumerate(range(n_test))  # [修改] enumerate 获取 job_index
    )

    p_values = np.array(p_values)

    # 4. 计算拒绝率
    fp = [pval < alpha for pval in p_values]
    final_result = np.mean(fp)
    fp1 = [pval < alpha1 for pval in p_values]
    final_result1 = np.mean(fp1)

    # 5. 打印结果
    print(f"\n" + "=" * 50)
    print(f"实验类型: {test.upper()}")
    print(f"Z Dimension: {z_dim}")
    print(f"Emp Rej Rate: {final_result:.4f} (at significance level {alpha})")
    print(f"Emp Rej Rate: {final_result1:.4f} (at significance level {alpha1})")
    print("=" * 50 + "\n")

    return p_values

In [8]:
param["sample_size"] = 500
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_mean"] = 0
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.15, 0.20]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[15:14:01] 开始并行实验...
模式: type1error, 样本量: 500, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.2850 (at significance level 0.1)
Emp Rej Rate: 0.1750 (at significance level 0.05)

[15:19:07] 开始并行实验...
模式: power, 样本量: 500, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.02

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1500 (at significance level 0.03086913086913087)
Emp Rej Rate: 0.0700 (at significance level 0.014885114885114887)

[15:23:56] 开始并行实验...
模式: power, 样本量: 500, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.04

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.0950 (at significance level 0.03086913086913087)
Emp Rej Rate: 0.0350 (at significance level 0.014885114885114887)

[15:28:47] 开始并行实验...
模式: power, 样本量: 500, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.06

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1600 (at significance 

In [14]:
# ============================================================
# 参数配置
# ============================================================
param = {
    "test": "power",            # ['type1error', 'power']
    "sample_size": 400,           # 推荐先用 400；算力充足可用 500/600/800/1000
    "batch_size": 128,            # [32, 64, 128, 256]
    "z_dim": 30,                  # 新 DGP 建议 dz=30 或 50；至少需要 dz>=3
    "dx": 1,
    "dy": 1,
    "n_test": 100,                # Monte Carlo repetitions；正式实验建议 >=200
    "epochs_num": 100,            # quick run 可用 100；正式实验可加大
    "eps_std": 1.0,               # residual multiplier；DGP 内部基础 sigma=0.6
    "dist_z": 'gaussian',         # ['gaussian', 'laplace', 'uniform']; DGP 固定为 mean_anchor
    "alpha_x": 0.05,              # H1 共享 residual 概率 p，建议 [0.03,0.05,0.075,0.10,0.15,0.20]
    "m_value": 100,               # Monte Carlo synthetic samples M
    "k_value": 2,                 # cross-fitting folds；论文和推荐实验通常 J=2 power 较高
    "j_value": 1000,              # bootstrap replicates B
    "noise_dimension": 5,         # generator latent noise dimension
    "noise_dimension_type": "normal",   # ["normal", "unif", "Cauchy"]
    "noise_dimension_var": 1,
    "hidden_layer_size": 1024,
    "normal_ini": False,
    "preprocess": 'scale',        # 当前 DGP 中保留参数但不做额外 preprocessing
    "G_lr": 7e-5,
    "alpha": 0.1,
    "alpha1": 0.05,
    "set_seeds": 0,
    "using_orcale": False,
    "lambda_1": 1,
    "lambda_2": 1,
    "using_Gen": '1',
    "boor_rv_type": 'gaussian',
    "wgt_decay": 1e-5,
    "lambda_3": 1e-4,
    "lambda_4": 2e-5,
    "drop_out_p": 0.2,
    "is_sparse": True,
    "sparse_ratio": 0.05,
    "lambda_mean": 0.3,           # mean alignment strength；baseline 设置为 0
    "mean_samples": 20
}


import torch
import torch.distributions as TD
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from datetime import datetime
import functools
from tqdm import tqdm


# ============================================================
# [修改] 删除全局 device 变量，改为在 mGAN() 内部按 gpu_id 动态创建。
# 原代码：
#   enable_cuda = True
#   device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')
# ============================================================



def _draw_z_for_mean_anchor(num, dz, dist_z='gaussian'):
    """
    Draw high-dimensional conditioning variable Z for the proposed mean-anchor DGP.


    The proposed DGP is sparse in Z: only Z1, Z2, Z3 enter the conditional means;
    the remaining coordinates are nuisance dimensions. Therefore dz should be at
    least 3, and dz=30 or dz=50 is recommended to highlight the benefit of mean
    alignment in high-dimensional conditioning.
    """
    if dz < 3:
        raise ValueError

    dist_z = str(dist_z).lower()


    if dist_z in ['mean_anchor', 'mean_nonlinear', 'gaussian', 'normal']:
        Z = torch.randn(num, dz)
    elif dist_z == 'laplace':
        laplace = TD.Laplace(torch.zeros(dz), torch.ones(dz))
        Z = laplace.sample((num,))
    elif dist_z == 'uniform':
        Z = 2.0 * torch.rand(num, dz) - 1.0
    else:
        raise ValueError(
            f"Unsupported dist_z: {dist_z}. Use 'gaussian', 'laplace', or 'uniform'."
        )


    return Z.float()



def _mean_anchor_conditional_means(Z, amp=2.0, use_bump=False):
    """
    Nonlinear sparse conditional means for the proposed DGP.


    S = (Z1 + Z2) / sqrt(2)
    m_X(Z) = amp * [sin(2S) + 0.5S + 0.4 tanh(Z3)]
    m_Y(Z) = amp * [cos(2S) - exp(-2) + 0.5S - 0.4 tanh(Z3)]


    If use_bump=True, an optional local bump is added to make the mean surface harder.
    """
    if Z.dim() == 1:
        Z = Z.reshape(-1, 1)
    if Z.shape[1] < 3:
        raise ValueError("Z must have at least 3 columns for the proposed mean-anchor DGP.")


    sqrt2 = np.sqrt(2.0)
    S = (Z[:, 0:1] + Z[:, 1:2]) / sqrt2
    Z3 = Z[:, 2:3]


    base_x = torch.sin(2.0 * S) + 0.5 * S + 0.4 * torch.tanh(Z3)
    base_y = torch.cos(2.0 * S) - np.exp(-2.0) + 0.5 * S - 0.4 * torch.tanh(Z3)


    if use_bump:
        bump = torch.exp(-8.0 * (S - 1.0) ** 2)
        base_x = base_x + bump
        base_y = base_y - bump


    mu_x = amp * base_x
    mu_y = amp * base_y


    return mu_x.float(), mu_y.float()



def generate_samples_random(size=1000, sType='H0', dx=1, dy=1, dz=30,
                            nstd=1.0, alpha_x=0.05,
                            preprocess="None", dist_z='gaussian'):
    """
    Proposed sparse high-dimensional nonlinear location-shift DGP.


    Purpose:
        This DGP is designed to highlight the gain from adding mean alignment to
        the original MMD/GMMN objective. Most of the nuisance structure is in the
        nonlinear conditional means E[X|Z] and E[Y|Z], while the conditional shape
        is simple Gaussian location shift.


    Z:
        Z in R^dz, dz recommended as 30 or 50.
        Only Z1, Z2, Z3 are signal coordinates; the rest are nuisance coordinates.


    H0:
        Y = m_Y(Z) + sigma * eps_Y
        X = m_X(Z) + sigma * eps_X
        eps_X independent of eps_Y, so X independent of Y given Z.


    H1:
        Y = m_Y(Z) + sigma * eps_Y
        X = m_X(Z) + sigma * [(1 - Delta) eps_X + Delta eps_Y]
        Delta ~ Bernoulli(alpha_x).


        Therefore Cov(X, Y | Z) = sigma^2 * alpha_x, while the conditional
        marginals X|Z and Y|Z remain the same as under H0.


    Parameters:
        alpha_x: dependence strength p under H1.
        nstd: residual multiplier. The base sigma is fixed at 0.6, so the actual
              residual sd is 0.6 * nstd.
    """
    num = size
    amp = 2.0
    base_sigma = 0.6
    sigma = base_sigma * nstd


    Z = _draw_z_for_mean_anchor(num, dz, dist_z=dist_z)
    mu_x, mu_y = _mean_anchor_conditional_means(Z, amp=amp, use_bump=False)


    eps_x = torch.randn(num, 1)
    eps_y = torch.randn(num, 1)


    if sType == 'H0':
        X = mu_x + sigma * eps_x
        Y = mu_y + sigma * eps_y


    elif sType == 'H1':
        delta = torch.bernoulli(torch.ones(num, 1) * alpha_x)
        X = mu_x + sigma * ((1.0 - delta) * eps_x + delta * eps_y)
        Y = mu_y + sigma * eps_y


    else:
        raise ValueError("sType must be 'H0' or 'H1'.")


    if dx > 1:
        X = X.repeat(1, dx)
    if dy > 1:
        Y = Y.repeat(1, dy)


    return X.float(), Y.float(), Z.float()



def generate_samples_from_fixed_Z_random(Z, size=1000, sType='H0',
                                          dx=1, dy=1, dz=30,
                                          nstd=1.0, alpha_x=0.05,
                                          normalize=True, seed=None,
                                          dist_z='gaussian'):
    """
    Oracle conditional generator for the proposed DGP.


    Important:
        The oracle generator must sample from the marginal conditional laws
        P_{X|Z} and P_{Y|Z}. Since the proposed H1 only introduces dependence by
        sharing residuals between X and Y, the marginal conditional distributions
        X|Z and Y|Z are unchanged under H0 and H1. Therefore this oracle generator
        always draws independent residuals for fake X and fake Y.
    """
    if seed is not None:
        torch.manual_seed(seed)


    if Z.dim() == 1:
        Z = Z.reshape(-1, 1)


    num = Z.shape[0]
    device = Z.device
    amp = 2.0
    base_sigma = 0.6
    sigma = base_sigma * nstd


    mu_x, mu_y = _mean_anchor_conditional_means(Z, amp=amp, use_bump=False)
    mu_x = mu_x.to(device)
    mu_y = mu_y.to(device)


    eps_x = torch.randn(num, 1, device=device)
    eps_y = torch.randn(num, 1, device=device)


    X = mu_x + sigma * eps_x
    Y = mu_y + sigma * eps_y


    if dx > 1:
        X = X.repeat(1, dx)
    if dy > 1:
        Y = Y.repeat(1, dy)


    return X.float(), Y.float()


def get_p_value_stat_1(boot_num, M, n, gen_x_all_torch, gen_y_all_torch,
                        x_torch, y_torch, z_torch, sigma_w,
                        sigma_u=1, sigma_v=1, boor_rv_type="gaussian",
                        device=None):  # [修改] 新增 device 参数
    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1),
        ord=1, dim=2)
    w_mx = torch.exp(-w_mx / sigma_w)


    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(torch.exp(-torch.abs(
        gen_y_all_torch.repeat(n, 1, 1) - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u), dim=2)
    u_mx_3 = u_mx_2.T
    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(torch.exp(-torch.abs(
        gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)


    v_mx_1 = torch.exp(-torch.abs(x_torch.repeat(1, n) - x_torch.repeat(1, n).T) / sigma_v)
    v_mx_2 = torch.mean(torch.exp(-torch.abs(
        gen_x_all_torch.repeat(n, 1, 1) - x_torch.repeat(1, n).reshape(n, n, 1)) / sigma_v), dim=2)
    v_mx_3 = v_mx_2.T
    gen_x_all_torch_rep = gen_x_all_torch.repeat(n, 1, 1)
    temp2_mx = gen_x_all_torch_rep[:, :, 0].T
    sum2_mx = torch.mean(torch.exp(-torch.abs(
        gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)


    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        temp_add_mx = torch.mean(torch.exp(-torch.abs(
            gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)
        sum_mx = sum_mx + temp_add_mx
        temp2_mx = gen_x_all_torch_rep[:, :, i].T
        temp2_add_mx = torch.mean(torch.exp(-torch.abs(
            gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)
        sum2_mx = sum2_mx + temp2_add_mx


    u_mx_4 = 1 / M * sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4
    v_mx_4 = 1 / M * sum2_mx
    v_mx = v_mx_1 - v_mx_2 - v_mx_3 + v_mx_4


    FF_mx = u_mx * v_mx * w_mx * (1 - torch.eye(n).to(device))  # [修改]
    stat = 1 / (n - 1) * torch.sum(FF_mx).item()


    boottemp = np.array([])
    if boor_rv_type == "rademacher":
        eboot = torch.sign(torch.randn(n, boot_num)).to(device)  # [修改]
    elif boor_rv_type == "gaussian":
        eboot = torch.randn(n, boot_num).to(device)              # [修改]
    for bb in range(boot_num):
        random_mx = torch.matmul(eboot[:, bb].reshape(-1, 1), eboot[:, bb].reshape(-1, 1).T)
        bootmatrix = FF_mx * random_mx
        stat_boot = 1 / (n - 1) * torch.sum(bootmatrix).item()
        boottemp = np.append(boottemp, stat_boot)
    return stat, boottemp



class DatasetSelect(Dataset):
    def __init__(self, X, Y, Z):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.sample_size = X.shape[0]


    def __len__(self):
        return self.sample_size


    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index]



class DatasetSelect_GAN(torch.utils.data.Dataset):
    def __init__(self, X, Y, Z, batch_size):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = X.shape[0]


    def __len__(self):
        return self.sample_size


    def __getitem__(self, index):
        return (self.X_real[index], self.Y_real[index], self.Z_real[index],
                self.Z_real[(self.batch_size + index) % self.sample_size])



class DatasetSelect_GAN_ver2(torch.utils.data.Dataset):
    def __init__(self, Y, Z, batch_size):
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = Z.shape[0]


    def __len__(self):
        return self.sample_size


    def __getitem__(self, index):
        return self.Y_real[index], self.Z_real[index]



##### Auxilliary functions #####


def sample_noise(sample_size, noise_dimension, noise_type, input_var=1,
                 device=None):  # [修改] 新增 device 参数
    if noise_type == "normal":
        noise_generator = TD.MultivariateNormal(
            torch.zeros(noise_dimension).to(device),          # [修改]
            input_var * torch.eye(noise_dimension).to(device) # [修改]
        )
        Z = noise_generator.sample((sample_size,))
    if noise_type == "unif":
        Z = torch.rand(sample_size, noise_dimension)
    if noise_type == "Cauchy":
        Z = TD.Cauchy(torch.tensor([0.0]), torch.tensor([1.0])).sample(
            (sample_size, noise_dimension)).squeeze(2)
    return Z


##### GAN architecture #####


class Generator(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef, drop_out_p, drop_input=False):
        super(Generator, self).__init__()
        self.BN_type = BN_type
        self.ReLU_coef = ReLU_coef
        self.fc1 = torch.nn.Linear(input_dimension + noise_dimension, hidden_layer_size, bias=True)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN2 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN3 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
        self.leakyReLU1 = torch.nn.LeakyReLU(ReLU_coef)
        self.fc2 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc3 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc_last = torch.nn.Linear(hidden_layer_size, output_dimension, bias=True)
        self.sigmoid = torch.nn.Sigmoid()
        self.drop_out0 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out1 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out2 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out3 = torch.nn.Dropout(p=drop_out_p)
        self.drop_input = drop_input


    def forward(self, x):
        if self.BN_type:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.BN1(self.fc1(x))))
            x = self.drop_out2(self.leakyReLU1(self.BN2(self.fc2(x))))
            # x = self.drop_out3(self.leakyReLU1(self.BN3(self.fc3(x))))
            x = self.fc_last(x)
        else:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.fc1(x)))
            x = self.drop_out2(self.leakyReLU1(self.fc2(x)))
            # x = self.drop_out3(self.leakyReLU1(self.fc3(x)))
            x = self.fc_last(x)
        return x



class NonFullyConnected_1(torch.nn.Module):
    def __init__(self, size_in, size_out, m, bias=True):
        super(NonFullyConnected_1, self).__init__()
        # [修改] 不在此处 .to(device)，由外层调用者统一 .to(device)
        self.linear = torch.nn.Linear(m * size_in, m * size_out, bias=bias)
        self.mask = functools.reduce(
            torch.block_diag,
            [torch.ones(size_out, size_in) for i in range(m)])


    def forward(self, x):
        # [修改] mask 动态跟随模型所在设备
        self.linear.weight.data *= self.mask.to(self.linear.weight.device)
        return self.linear(x)



class Generator_2(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef,
                 hidden_layer_depth=1, ntargets_k=5):
        super(Generator_2, self).__init__()
        self.input_dimension = input_dimension + noise_dimension
        self.output_dimension = output_dimension
        self.ntargets_k = ntargets_k
        self.hidden_layer_sizes = [hidden_layer_size] * hidden_layer_depth
        self.BN_type = BN_type
        self.leakyrelu = torch.nn.LeakyReLU(ReLU_coef)
        self.linear_layers_from_input = torch.nn.Linear(
            self.input_dimension, ntargets_k * self.hidden_layer_sizes[0])
        self.linear_layers_between = torch.nn.ModuleList([
            NonFullyConnected_1(self.hidden_layer_sizes[0], self.hidden_layer_sizes[0], ntargets_k)
            for i in range(len(self.hidden_layer_sizes))
        ])
        self.linear8 = torch.nn.Linear(
            self.hidden_layer_sizes[0] * ntargets_k, self.output_dimension)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)


    def forward(self, input):
        if self.BN_type:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(self.BN1(output))
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(self.BN1(output))
        else:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(output)
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(output)
        return self.linear8(output)



##### Training procedures #####


def find_loss(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-torch.abs(Y.repeat(1, n) - Y.repeat(1, n).T) / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    mx_1_2 = torch.exp(-mx_1_2 / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2_1 = torch.exp(-torch.abs(Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y)
    mx_2 = mx_2_1 * mx_1_2
    mx_3 = mx_2.T
    mx_4_1 = torch.exp(-torch.abs(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y)
    mx_4 = mx_4_1 * mx_1_2
    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss = 1 / (n ** 2) * torch.sum(FF_mx)
    return loss



def find_loss_2(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-(Y.repeat(1, n) - Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=2, dim=2)
    mx_1_2 = torch.exp(-(mx_1_2 ** 2) / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2_1 = torch.exp(-(Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_2 = mx_2_1 * mx_1_2
    mx_3 = mx_2.T
    mx_4_1 = torch.exp(-(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_4 = mx_4_1 * mx_1_2
    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss = 1 / (n ** 2) * torch.sum(FF_mx)
    return loss



def train_ver3(
        G_zx, G_zy,
        X, Y, Z, X_test, Y_test, Z_test, M,
        noise_dimension, noise_type, G_lr, hidden_layer_size,
        DataLoader, BN_type, ReLU_coef,
        lambda_mean=0.5, mean_samples=20,
        epochs_num=50, patience=5, min_delta=1e-5,
        sigma_z=1, sigma_x=1, sigma_y=1, normal_ini=False,
        lambda_1=1, lambda_2=1, using_Gen='1', wgt_decay=0,
        lambda_3=0, drop_out_p=0.2, noise_dimension_var=1, lambda_4=0,
        device=None):  # [修改] 新增 device 参数
    input_dimension = Z.shape[1]
    output_dimension_y = Y.shape[1]
    output_dimension_x = X.shape[1]


    if G_zy is None or G_zx is None:
        if using_Gen == '1':
            G_zy = Generator(input_dimension, output_dimension_y, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)  # [修改]
            G_zx = Generator(input_dimension, output_dimension_x, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)  # [修改]
        elif using_Gen == '2':
            G_zy = Generator_2(input_dimension, output_dimension_y, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)             # [修改]
            G_zx = Generator_2(input_dimension, output_dimension_x, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)             # [修改]
        if normal_ini:
            for p in G_zy.parameters():
                p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))  # [修改]
            for p in G_zx.parameters():
                p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))  # [修改]


    # 无论是否是新模型，每一折都重新定义优化器以清空动量状态
    G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
    G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)


    iter_count = 0
    G_zy = G_zy.train()
    G_zx = G_zx.train()


    best_loss = float('inf')
    counter = 0


    for epoch in range(epochs_num):
        batch_count = 0
        G_zy = G_zy.train()
        G_zx = G_zx.train()


        for X_real, Y_real, Z_real, Z_fake in DataLoader:
            X_real = X_real.to(device)   # [修改]
            Y_real = Y_real.to(device)   # [修改]
            Z_real = Z_real.to(device)   # [修改]
            Z_fake = Z_fake.to(device)   # [修改]


            batch_size = Z_real.shape[0]
            Z_repeated = Z_real.repeat_interleave(mean_samples, dim=0)


            Noise_for_mean = sample_noise(Z_repeated.shape[0], noise_dimension, noise_type,
                                          input_var=noise_dimension_var, device=device).to(device)  # [修改]


            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type,
                                      input_var=noise_dimension_var, device=device).to(device)       # [修改]
            Y_fake = G_zy(torch.cat((Z_real, Noise_fake), dim=1)).to(device)
            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type,
                                      input_var=noise_dimension_var, device=device).to(device)       # [修改]
            X_fake = G_zx(torch.cat((Z_real, Noise_fake), dim=1)).to(device)


            Y_generated_group = G_zy(torch.cat((Z_repeated, Noise_for_mean), dim=1))
            Y_generated_reshaped = Y_generated_group.reshape(batch_size, mean_samples, -1)
            Y_mean_pred = torch.mean(Y_generated_reshaped, dim=1)
            loss_mean_y = torch.nn.functional.mse_loss(Y_mean_pred, Y_real)


            X_generated_group = G_zx(torch.cat((Z_repeated, Noise_for_mean), dim=1))
            X_mean_pred = torch.mean(X_generated_group.reshape(batch_size, mean_samples, -1), dim=1)
            loss_mean_x = torch.nn.functional.mse_loss(X_mean_pred, X_real)


            g_zy_error = None
            G_zy_solver.zero_grad()
            g_zx_error = None
            G_zx_solver.zero_grad()


            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0
            for name, param in G_zy.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param, ord=1)


            mmd_loss_y = (lambda_1 * find_loss(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y) +
                          lambda_2 * find_loss_2(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y) +
                          lambda_3 * l1_regularization_rest_layers +
                          lambda_4 * l1_regularization_first_layer)
            g_zy_error = mmd_loss_y + lambda_mean * loss_mean_y
            g_zy_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zy.parameters(), max_norm=0.5)
            G_zy_solver.step()


            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0
            for name, param in G_zx.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param, ord=1)


            mmd_loss_x = (lambda_1 * find_loss(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x) +
                          lambda_2 * find_loss_2(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x) +
                          lambda_3 * l1_regularization_rest_layers +
                          lambda_4 * l1_regularization_first_layer)
            g_zx_error = mmd_loss_x + lambda_mean * loss_mean_x
            g_zx_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zx.parameters(), max_norm=0.5)
            G_zx_solver.step()


            iter_count += 1
            batch_count += 1


        current_loss = g_zx_error + g_zy_error
        if current_loss < best_loss - min_delta:
            best_loss = current_loss
            counter = 0
        else:
            counter += 1
        if counter >= patience:
            break


    return G_zy, G_zx



def mGAN(n=500, z_dim=1, simulation='type1error', batch_size=64, epochs_num=1000,
         nstd=1.0, z_dist='gaussian', x_dims=1, y_dims=1, a_x=0.05, M=500, k=2, boot_num=1000,
         noise_dimension=10, hidden_layer_size=512, normal_ini=False, preprocess='normalize',
         G_lr=1e-5, using_orcale=False, lambda_1=1, lambda_2=1,
         lambda_mean=0.3, mean_samples=20, using_Gen='1',
         boor_rv_type="gaussian", wgt_decay=0, lambda_3=1, drop_out_p=0.2,
         noise_dimension_var=1, noise_dimension_type="normal", lambda_4=1,
         gpu_id=0):
    """
    修正版 mGAN 函数。


    主要修正：
    1. 正确使用 median heuristic 计算 Laplacian kernel bandwidth：
       sigma_w = median_{i != j} ||Z_i - Z_j||_1
       sigma_u = median_{i != j} ||Y_i - Y_j||_1
       sigma_v = median_{i != j} ||X_i - X_j||_1


    2. 不再使用 median(exp(-distance)) 作为 bandwidth。


    3. 对 p-value 计算加入 tie correction：
       p = (1 + #{bootstrap_stat >= observed_stat}) / (B + 1)
       这样当 bootstrap statistic 与 observed statistic 出现大量 tie 时，
       不会错误地产生 p-value = 0。
    """


    # ============================================================
    # 0. 内部 helper：正确的 median L1 bandwidth
    # ============================================================


    def median_l1_bandwidth(A, min_bw=1e-6):
        """
        A: tensor with shape n x d
        return: median_{i != j} ||A_i - A_j||_1


        注意：
        原来的代码错误地使用 median(exp(-distance))。
        这里使用的是 pairwise distance 的 median。
        """


        n_local = A.shape[0]


        dist = torch.linalg.vector_norm(
            A.repeat(n_local, 1, 1) - torch.swapaxes(A.repeat(n_local, 1, 1), 0, 1),
            ord=1,
            dim=2
        )


        mask = ~torch.eye(n_local, dtype=torch.bool, device=A.device)
        vals = dist[mask]
        vals = vals[vals > 0]


        if vals.numel() == 0:
            return min_bw


        bw = torch.median(vals).item()


        if (not np.isfinite(bw)) or bw < min_bw:
            bw = min_bw


        return bw


# ============================================================
    # 1. 设置 device
    # ============================================================


    enable_cuda = True


    if torch.cuda.is_available() and enable_cuda:
        device = torch.device(f'cuda:{gpu_id}')
    else:
        device = torch.device('cpu')


    # ============================================================
    # 2. 生成数据
    # ============================================================


    if simulation == 'type1error':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n,
            sType='H0',
            dx=x_dims,
            dy=y_dims,
            dz=z_dim,
            nstd=nstd,
            alpha_x=a_x,
            dist_z=z_dist,
            preprocess=preprocess
        )


    elif simulation == 'power':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n,
            sType='H1',
            dx=x_dims,
            dy=y_dims,
            dz=z_dim,
            nstd=nstd,
            alpha_x=a_x,
            dist_z=z_dist,
            preprocess=preprocess
        )


    else:
        raise ValueError('Test does not exist. simulation must be type1error or power.')


    x = sim_x.to(device)
    y = sim_y.to(device)
    z = sim_z.to(device)


    # ============================================================
    # 3. 修正后的 bandwidth 计算
    # ============================================================
    # 原来错误写法：
    #
    #   u_mx = torch.exp(-torch.abs(y.repeat(1, n) - y.repeat(1, n).T))
    #   sigma_u = torch.median(u_mx).item()
    #
    #   v_mx = torch.exp(-torch.abs(x.repeat(1, n) - x.repeat(1, n).T))
    #   sigma_v = torch.median(v_mx).item()
    #
    # 现在改成 pairwise distance 的 median。


    sigma_w = median_l1_bandwidth(z)
    sigma_u = median_l1_bandwidth(y)
    sigma_v = median_l1_bandwidth(x)


    # 如果你想 debug，可以临时打开这行：
    # print(f"sigma_w={sigma_w:.6f}, sigma_u={sigma_u:.6f}, sigma_v={sigma_v:.6f}")


    # ============================================================
    # 4. 初始化 cross-fitting 容器
    # ============================================================


    test_size = int(n / k)


    stat_all = torch.zeros(k, 1)
    boot_temp_all = torch.zeros(k, boot_num)


    cur_k = 0


    # ============================================================
    # 5. 初始化 generator
    # ============================================================
    # 保持你原来的写法：generator 在 fold 外初始化，fold 间继续训练。
    # 严格 cross-fitting 下建议每个 fold 重新初始化 generator；
    # 但本函数根据你的要求只重点修正 bandwidth 问题。


    if not using_orcale:
        G_zy = Generator(
            z_dim,
            y_dims,
            noise_dimension,
            hidden_layer_size,
            False,
            0.1,
            drop_out_p
        ).to(device)


        G_zx = Generator(
            z_dim,
            x_dims,
            noise_dimension,
            hidden_layer_size,
            False,
            0.1,
            drop_out_p
        ).to(device)
    else:
        G_zy = None
        G_zx = None


    # ============================================================
    # 6. Cross-fitting loop
    # ============================================================


    for k_fold in range(k):


        k_fold_start = int(n / k * k_fold)
        k_fold_end = int(n / k * (k_fold + 1))


        X_test = x[k_fold_start:k_fold_end]
        Y_test = y[k_fold_start:k_fold_end]
        Z_test = z[k_fold_start:k_fold_end]


        X_train = torch.cat((x[0:k_fold_start], x[k_fold_end:]))
        Y_train = torch.cat((y[0:k_fold_start], y[k_fold_end:]))
        Z_train = torch.cat((z[0:k_fold_start], z[k_fold_end:]))


        if k == 1:
            X_train, Y_train, Z_train = X_test, Y_test, Z_test


        train_xyz = DatasetSelect_GAN(X_train, Y_train, Z_train, batch_size)


        DataLoader_xyz = torch.utils.data.DataLoader(
            train_xyz,
            batch_size=batch_size,
            shuffle=True
        )


        # ========================================================
        # 6.1 训练 generator
        # ========================================================


        if not using_orcale:


            G_zy, G_zx = train_ver3(
                G_zx=G_zx,
                G_zy=G_zy,
                X=X_train,
                Y=Y_train,
                Z=Z_train,
                M=M,
                X_test=X_test,
                Y_test=Y_test,
                Z_test=Z_test,
                noise_dimension=noise_dimension,
                noise_type=noise_dimension_type,
                G_lr=G_lr,
                hidden_layer_size=hidden_layer_size,
                DataLoader=DataLoader_xyz,
                BN_type=False,
                ReLU_coef=0.1,
                lambda_mean=lambda_mean,
                mean_samples=mean_samples,
                epochs_num=epochs_num,
                sigma_z=sigma_w,
                sigma_x=sigma_v,
                sigma_y=sigma_u,
                normal_ini=normal_ini,
                lambda_1=lambda_1,
                lambda_2=lambda_2,
                using_Gen=using_Gen,
                wgt_decay=wgt_decay,
                lambda_3=lambda_3,
                drop_out_p=drop_out_p,
                noise_dimension_var=noise_dimension_var,
                lambda_4=lambda_4,
                device=device
            )


        # ========================================================
        # 6.2 在 test fold 上生成 fake samples
        # ========================================================


        dataset_test = DatasetSelect(X_test, Y_test, Z_test)


        dataloader_test = DataLoader(
            dataset_test,
            batch_size=1,
            shuffle=True
        )


        gen_x_all = torch.zeros(test_size, M, device=device)
        gen_y_all = torch.zeros(test_size, M, device=device)


        z_all = torch.zeros(test_size, z_dim, device=device)
        x_all = torch.zeros(test_size, x_dims, device=device)
        y_all = torch.zeros(test_size, y_dims, device=device)


        cur_itr = 0


        if not using_orcale:
            G_zx = G_zx.eval()
            G_zy = G_zy.eval()


        for i, (x_test, y_test, z_test) in enumerate(dataloader_test):


            x_test = x_test.to(device)
            y_test = y_test.to(device)
            z_test = z_test.to(device)


            z_test_temp = z_test.repeat(M, 1)


            if not using_orcale:


                z_test_temp = z_test_temp.to(device)


                Noise_fake_x = sample_noise(
                    z_test_temp.size()[0],
                    noise_dimension,
                    noise_dimension_type,
                    input_var=noise_dimension_var,
                    device=device
                ).to(device)


                with torch.no_grad():
                    fake_x = G_zx(
                        torch.cat((z_test_temp, Noise_fake_x), dim=1)
                    ).reshape(1, -1)


                Noise_fake_y = sample_noise(
                    z_test_temp.size()[0],
                    noise_dimension,
                    noise_dimension_type,
                    input_var=noise_dimension_var,
                    device=device
                ).to(device)


                with torch.no_grad():
                    fake_y = G_zy(
                        torch.cat((z_test_temp, Noise_fake_y), dim=1)
                    ).reshape(1, -1)


            elif using_orcale:


                if simulation == 'type1error':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(
                        z_test_temp,
                        size=M,
                        sType='H0',
                        dx=x_dims,
                        dy=y_dims,
                        dz=z_dim,
                        nstd=nstd,
                        alpha_x=a_x,
                        dist_z=z_dist
                    )


                elif simulation == 'power':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(
                        z_test_temp,
                        size=M,
                        sType='H1',
                        dx=x_dims,
                        dy=y_dims,
                        dz=z_dim,
                        nstd=nstd,
                        alpha_x=a_x,
                        dist_z=z_dist
                    )


                fake_x = fake_x.to(device).reshape(1, -1)
                fake_y = fake_y.to(device).reshape(1, -1)


            gen_x_all[cur_itr, :] = fake_x.detach().reshape(-1)
            gen_y_all[cur_itr, :] = fake_y.detach().reshape(-1)


            x_all[cur_itr, :] = x_test
            y_all[cur_itr, :] = y_test
            z_all[cur_itr, :] = z_test


            cur_itr = cur_itr + 1


        # ========================================================
        # 6.3 计算 statistic 和 bootstrap statistic
        # ========================================================


        cur_stat, cur_boot_temp = get_p_value_stat_1(
            boot_num,
            M,
            test_size,
            gen_x_all.to(device),
            gen_y_all.to(device),
            x_all.to(device),
            y_all.to(device),
            z_all.to(device),
            sigma_w,
            sigma_u,
            sigma_v,
            boor_rv_type,
            device=device
        )


        stat_all[cur_k, :] = cur_stat
        boot_temp_all[cur_k, :] = torch.from_numpy(cur_boot_temp)


        cur_k = cur_k + 1


    # ============================================================
    # 7. Oracle diagnostic
    # ============================================================


    if using_orcale:
        gen_x_mean = torch.mean(gen_x_all, dim=1).reshape(-1, 1)
        gen_y_mean = torch.mean(gen_y_all, dim=1).reshape(-1, 1)


        mse_x = torch.mean((gen_x_mean - x_all) ** 2).item()
        mse_y = torch.mean((gen_y_mean - y_all) ** 2).item()


        print(f'Test MSE x [{mse_x}], MSE y [{mse_y}]')


    # ============================================================
    # 8. p-value
    # ============================================================
    # 原代码：
    #
    #   return np.mean(torch.mean(boot_temp_all, dim=0).numpy() > torch.mean(stat_all).item())
    #
    # 这里顺手改成 tie-corrected p-value。
    # 如果 bootstrap statistic 和 observed statistic 大量相等，
    # 原来的严格 > 会把 p-value 错误算成 0。


    boot_stats = torch.mean(boot_temp_all, dim=0).detach().cpu().numpy()
    obs_stat = torch.mean(stat_all).item()


    p_value = (1.0 + np.sum(boot_stats >= obs_stat)) / (len(boot_stats) + 1.0)


    return p_value
    
# ============================================================
# 并行部分
# ============================================================
from joblib import Parallel, delayed
import multiprocessing
import numpy as np
from datetime import datetime


def run_experiment(params):
    # 1. 提取参数
    test                 = params["test"]
    sample_size          = params["sample_size"]
    batch_size           = params["batch_size"]
    z_dim                = params["z_dim"]
    dx                   = params["dx"]
    dy                   = params["dy"]
    n_test               = params["n_test"]
    epochs_num           = params["epochs_num"]
    eps_std              = params["eps_std"]
    dist_z               = params["dist_z"]
    alpha_x              = params["alpha_x"]
    m_value              = params["m_value"]
    k_value              = params["k_value"]
    j_value              = params["j_value"]
    noise_dimension      = params["noise_dimension"]
    noise_dimension_type = params["noise_dimension_type"]
    noise_dimension_var  = params["noise_dimension_var"]
    hidden_layer_size    = params["hidden_layer_size"]
    normal_ini           = params["normal_ini"]
    preprocess           = params["preprocess"]
    G_lr                 = params["G_lr"]
    alpha                = params["alpha"]
    alpha1               = params["alpha1"]
    set_seeds            = params["set_seeds"]
    using_orcale         = params["using_orcale"]
    lambda_1             = params["lambda_1"]
    lambda_2             = params["lambda_2"]
    using_Gen            = params["using_Gen"]
    boor_rv_type         = params["boor_rv_type"]
    wgt_decay            = params["wgt_decay"]
    lambda_3             = params["lambda_3"]
    lambda_4             = params["lambda_4"]
    drop_out_p           = params["drop_out_p"]
    lambda_mean          = params["lambda_mean"]
    mean_samples         = params["mean_samples"]


    np.random.seed(set_seeds)
    torch.manual_seed(set_seeds)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(set_seeds)


    # 2. [修改] 获取可用 GPU 数量，用于轮询分配
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    num_cores = min(28, n_test)
    if num_cores < 1:
        num_cores = 1


    print(f"[{datetime.now().strftime('%H:%M:%S')}] 开始并行实验...")
    print(f"模式: {test}, 样本量: {sample_size}, 交叉验证折数: {k_value}, "
          f"模型均值对齐超参数: {lambda_mean}, 实验次数: {n_test}, "
          f"并行核数: {num_cores}, 可用GPU数: {num_gpus}")
    if test == 'power':
        print(f"备择假设H_1下的模型参数 alpha_x: {alpha_x}")


    # 3. [修改] 按 job_index % num_gpus 轮询分配 GPU
    #    job 0,4,8,...  → GPU 0
    #    job 1,5,9,...  → GPU 1
    #    job 2,6,10,... → GPU 2
    #    job 3,7,11,... → GPU 3
    p_values = Parallel(n_jobs=num_cores)(
        delayed(mGAN)(
            n=sample_size, z_dim=z_dim, simulation=test, batch_size=batch_size,
            epochs_num=epochs_num, nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy,
            a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
            noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size,
            normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr,
            using_orcale=using_orcale, lambda_1=lambda_1, lambda_2=lambda_2,
            lambda_mean=lambda_mean, mean_samples=mean_samples,
            using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay,
            lambda_3=lambda_3, drop_out_p=drop_out_p,
            noise_dimension_var=noise_dimension_var,
            noise_dimension_type=noise_dimension_type, lambda_4=lambda_4,
            gpu_id=job_index % num_gpus    # [修改] 轮询分配 GPU
        )
        for job_index, _ in enumerate(range(n_test))  # [修改] enumerate 获取 job_index
    )


    p_values = np.array(p_values)


    # 4. 计算拒绝率
    fp = [pval < alpha for pval in p_values]
    final_result = np.mean(fp)
    fp1 = [pval < alpha1 for pval in p_values]
    final_result1 = np.mean(fp1)


    # 5. 打印结果
    print(f"\n" + "=" * 50)
    print(f"实验类型: {test.upper()}")
    print(f"Z Dimension: {z_dim}")
    print(f"Emp Rej Rate: {final_result:.4f} (at significance level {alpha})")
    print(f"Emp Rej Rate: {final_result1:.4f} (at significance level {alpha1})")
    print("=" * 50 + "\n")


    return p_values
if __name__ == "__main__":
    # ============================================================
    # 推荐实验入口：比较 baseline(lambda_mean=0) 与 mean alignment
    # ============================================================
    # 说明：
    # 1. H0 下先得到 empirical p-value 分布，用于 type-I error 与 size-adjusted threshold。
    # 2. H1 下使用同一个 lambda_mean 对应的 H0 empirical quantile 做 size-adjusted power。
    # 3. 若只想快速测试代码是否跑通，可把 n_test 改成 10、epochs_num 改成 20、j_value 改成 200。
    # 4. 正式实验建议 n_test >= 200，j_value = 1000，m_value = 100。


    base_param = param.copy()
    base_param["sample_size"] = 400
    base_param["z_dim"] = 30
    base_param["dist_z"] = "gaussian"
    base_param["k_value"] = 2
    base_param["m_value"] = 100
    base_param["j_value"] = 1000
    base_param["mean_samples"] = 30
    base_param["n_test"] = 100


    lambda_grid = [0.0, 0.1, 0.5]
    alpha_x_grid = [0.03, 0.05, 0.075, 0.10, 0.15, 0.20]


    all_results = {}


    for lambda_mean_value in lambda_grid:
        print("\n" + "#" * 80)
        print(f"开始 lambda_mean = {lambda_mean_value} 的实验")
        print("#" * 80)


        cur_param = base_param.copy()
        cur_param["lambda_mean"] = lambda_mean_value


        # --------------------------------------------------------
        # H0: type-I error and empirical calibration thresholds
        # --------------------------------------------------------
        cur_param["test"] = "type1error"
        cur_param["alpha"] = 0.10
        cur_param["alpha1"] = 0.05
        cur_param["alpha_x"] = 0.0


        p_val_h0 = run_experiment(cur_param)
        all_results[(lambda_mean_value, "H0")] = p_val_h0


        alpha_emp_10 = np.quantile(p_val_h0, 0.10)
        alpha_emp_05 = np.quantile(p_val_h0, 0.05)


        print(f"lambda_mean={lambda_mean_value}: empirical threshold for size 0.10 = {alpha_emp_10:.6f}")
        print(f"lambda_mean={lambda_mean_value}: empirical threshold for size 0.05 = {alpha_emp_05:.6f}")


        # --------------------------------------------------------
        # H1: size-adjusted power curve
        # --------------------------------------------------------
        cur_param["test"] = "power"
        cur_param["alpha"] = alpha_emp_10
        cur_param["alpha1"] = alpha_emp_05


        for alpha_x in alpha_x_grid:
            cur_param["alpha_x"] = alpha_x
            p_val_h1 = run_experiment(cur_param)
            all_results[(lambda_mean_value, f"H1_alpha_x_{alpha_x}")] = p_val_h1


            power_10 = np.mean(p_val_h1 < alpha_emp_10)
            power_05 = np.mean(p_val_h1 < alpha_emp_05)
            print(
                f"lambda_mean={lambda_mean_value}, alpha_x={alpha_x}: "
                f"size-adjusted power@0.10={power_10:.4f}, "
                f"size-adjusted power@0.05={power_05:.4f}"
            )


    # 保存所有 p-values，便于后续画 power curve 或制表。
    np.savez("mean_anchor_dgp_results.npz", **{str(k): v for k, v in all_results.items()})
    print("\n所有实验完成，p-values 已保存到 mean_anchor_dgp_results.npz")

SyntaxError: invalid non-printable character U+00A0 (3580101512.py, line 5)